# 03 — Conectar busca e geração: o primeiro RAG

As etapas 01 e 02 prepararam documentos rastreáveis e um índice persistente. Nesta etapa, a pergunta é codificada pelo encoder, usada para recuperar trechos no Chroma e enviada ao Qwen junto com esses trechos. Esse padrão é chamado RAG (geração aumentada por recuperação).

```mermaid
flowchart LR
 A[pergunta] --> B[encoder de consulta]
 B --> C[Chroma / HNSW]
 C --> D[trechos + páginas]
 A --> E[prompt]
 D --> E
 E --> F[Qwen 3B]
 F --> G[resposta + fontes]
```

O modelo e seus pesos são os mesmos da etapa 00. A mudança é a inclusão de evidências no prompt: mede-se se a recuperação ajuda a resposta a se apoiar no corpus, sem presumir que isso a torne automaticamente correta.

In [1]:
# Objetivo: definir diretórios do fluxo RAG.
from pathlib import Path
import json

RANDOM_STATE = 42
DIRETORIO_PROJETO = Path.cwd()
if DIRETORIO_PROJETO.name != "Agente_IA_Transformers_Credito":
    DIRETORIO_PROJETO = DIRETORIO_PROJETO / "Agente_IA_Transformers_Credito"
if not DIRETORIO_PROJETO.is_dir():
    raise FileNotFoundError("Inicie o kernel na raiz do repositório ou na pasta do projeto.")

DIRETORIO_DOCUMENTOS = DIRETORIO_PROJETO / "documentos"
DIRETORIO_PROCESSADOS = DIRETORIO_PROJETO / "dados_processados"
DIRETORIO_RESULTADOS = DIRETORIO_PROJETO / "resultados"
DIRETORIO_PROCESSADOS.mkdir(exist_ok=True)
DIRETORIO_RESULTADOS.mkdir(exist_ok=True)
print("Projeto:", DIRETORIO_PROJETO)

Projeto: d:\Projetos_v2\Agente_IA_Transformers_Credito


## 1. Recarregar os componentes persistidos

Este notebook pode começar em um kernel novo. Reabre-se a coleção criada na etapa 02, carrega-se o mesmo encoder da etapa 01 e, por fim, a LLM definida na etapa 00. O encoder fica na CPU para reservar a VRAM da RTX 2060 SUPER ao Qwen. Nesta comparação inicial ainda não há adaptador QLoRA: primeiro se observa a base com e sem recuperação.

Encoder, banco e gerador têm funções distintas: o encoder representa textos em vetores; o Chroma localiza trechos próximos da consulta; o Qwen redige uma resposta a partir da pergunta e do contexto recebido. Por exemplo, para “como funciona a atenção?”, a busca pode trazer passagens sobre Q, K e V, e o Qwen usa essas passagens ao explicar o tema.

Se o manifesto, a coleção ou o modelo não corresponder ao esperado, os artefatos não devem ser misturados: uma mudança de encoder ou de corpus altera a interpretação dos resultados.

In [2]:
# Objetivo: abrir Chroma e carregar encoder e Qwen.
import chromadb
from sentence_transformers import SentenceTransformer
from rag import ENCODER, COLECAO, carregar_llm, recuperar, montar_mensagens, gerar

# Reabre o banco persistido no caminho criado na etapa 02.
cliente = chromadb.PersistentClient(path=str(DIRETORIO_PROJETO / "banco_chroma"))
colecao = cliente.get_collection(COLECAO)
# Carrega o mesmo encoder do índice; device="cpu" deixa a VRAM disponível ao Qwen.
encoder = SentenceTransformer(ENCODER, device="cpu")
modelo, tokenizador = carregar_llm()
print("Registros disponíveis:", colecao.count())

c:\Users\leova\anaconda3\envs\engenharia_ia_venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 434/434 [00:29<00:00, 14.64it/s]


Registros disponíveis: 990


## 2. Buscar e ler os trechos antes de gerar

A função `recuperar` transforma a pergunta em embedding normalizado e consulta o Chroma. Antes de chamar a LLM, examinam-se título, página, distância e texto. Isso ajuda a localizar a origem de uma falha: o índice pode não trazer a evidência necessária, ou o Qwen pode receber a evidência e interpretá-la mal.

A distância apenas ordena os candidatos conforme o encoder. Suponha que a pergunta peça (1) a equação da atenção e (2) um exemplo PyTorch. Um trecho sobre a equação cobre só a primeira necessidade; se nenhum resultado contiver código, a recuperação não cobriu a pergunta, mesmo que o primeiro resultado esteja muito próximo. A inspeção deve considerar cobertura, procedência e conteúdo.

In [5]:
# Objetivo: recuperar e inspecionar evidências.
pergunta_rag = "Como a atenção pode representar sequências?"
# Recupera trechos relevantes; colecao e encoder devem corresponder ao corpus, k define a saída e fetch_k os candidatos MMR.
trechos = recuperar(
    colecao,  # Coleção Chroma que guarda textos e metadados de origem.
    encoder,  # Encoder compatível que cria o vetor da pergunta.
    pergunta_rag,  # Texto da consulta a ser respondida com apoio do corpus.
    usar_mmr=True,  # Ativa seleção diversificada para reduzir trechos redundantes.
    k=4,  # Quantidade de trechos devolvidos ao final.
    fetch_k=12,  # Candidatos recuperados antes da seleção MMR.
)
for trecho in trechos:
    print(trecho["source_id"], "página", trecho["page"])
    print(trecho["texto"][:500], "\n")

attention_is_all_you_need página 2
sequence in order to compute a representation of the sequence. Self-attention has been used successfully in a variety of tasks including reading comprehension, abstractive summarization, textual entailment and learning task-independent sentence representations [4, 27, 28, 22]. End-to-end memory networks are based on a recurrent attention mechanism instead of sequence- aligned recurrence and have been 

attention_is_all_you_need página 13
<EOS> Figure 3: An example of the attention mechanism following long-distance dependencies in the encoder self-attention in layer 5 of 6. Many of the attention heads attend to a distant dependency of the verb ‘making’, completing the phrase ‘making...more difficult’. Attentions here shown only for the word ‘making’. Different colors represent different heads. Best viewed in color. 13 

attention_is_all_you_need página 2
]. In the Transformer this is reduced to a constant number of operations, albeit at the cost of red

## 3. Gerar e conferir a resposta com evidências

`montar_mensagens` organiza a pergunta e os trechos recuperados no formato de conversa do Qwen. O prompt pede que as fontes sejam usadas, mas uma instrução não garante fidelidade: afirmações e citações ainda precisam ser conferidas no trecho e na página indicados.

RAG significa geração aumentada por recuperação. Os trechos são adicionados ao contexto de entrada; os pesos do Qwen não mudam. O resultado depende de duas etapas: recuperar evidência adequada e o gerador usá-la corretamente. Uma busca boa não garante uma boa resposta, e uma resposta plausível pode exceder o que as fontes sustentam.

### Orçamento de contexto

O limite desta implementação é de 3.072 tokens de entrada, e `gerar` permite até 512 novos tokens de saída. Instrução, histórico, pergunta e evidências compartilham o espaço de entrada. Exemplo ilustrativo para uma entrada de 1.024 tokens: instrução 150 + histórico 200 + pergunta 50 + evidências 400 + espaço restante 224. Se o texto crescer, será necessário reduzir histórico ou selecionar trechos antes da geração. Cortar sem inspeção pode remover a frase que apoiava a resposta.

Uma citação útil aponta para a origem efetivamente recuperada — identificador da fonte, página e trecho. Citar um documento apenas relacionado ao assunto não fundamenta uma afirmação.

In [6]:
# Objetivo: montar contexto e gerar resposta com o Qwen.
from rag import gerar
# Combina pergunta e evidências em mensagens compatíveis com a instrução RAG.
mensagens_rag = montar_mensagens(pergunta_rag, trechos)
print(mensagens_rag[-1]["content"])
# Executa a geração com modelo, tokenizador e mensagens que incluem o contexto recuperado.
resposta_rag = gerar(modelo, tokenizador, mensagens_rag)
print(resposta_rag)

EVIDÊNCIAS:
[attention_is_all_you_need, p. 2] sequence in order to compute a representation of the sequence. Self-attention has been used successfully in a variety of tasks including reading comprehension, abstractive summarization, textual entailment and learning task-independent sentence representations [4, 27, 28, 22]. End-to-end memory networks are based on a recurrent attention mechanism instead of sequence- aligned recurrence and have been

[attention_is_all_you_need, p. 13] <EOS> Figure 3: An example of the attention mechanism following long-distance dependencies in the encoder self-attention in layer 5 of 6. Many of the attention heads attend to a distant dependency of the verb ‘making’, completing the phrase ‘making...more difficult’. Attentions here shown only for the word ‘making’. Different colors represent different heads. Best viewed in color. 13

[attention_is_all_you_need, p. 2] ]. In the Transformer this is reduced to a constant number of operations, albeit at the cost

## 4. Levar o mesmo fluxo à interface

O notebook permite inspecionar separadamente consulta, busca, contexto e geração. A interface Streamlit envia cada mensagem à API FastAPI, que chama as mesmas funções compartilhadas e devolve resposta e fontes. Isso torna o fluxo consultável sem executar manualmente as células.

A interface oferece uma configuração de modelo por processo para respeitar a memória da GPU. O modo Ajustado só faz sentido depois de existir um adaptador treinado e de ele ter sido avaliado na etapa 05. A interface facilita uso; ela não substitui a verificação das evidências nem a avaliação das respostas.